# Background comparison: ecoinvent 3.12 vs BAFU-2026

Compares the results of the same foreground models with 2 background databases.

1. Run `python -m l4m_lca.bafu` one time. It makes the project `Lasers4MaaS_BAFU` and imports BAFU-2026.
2. Run `joining_task`, `pre_post` and `workflow_assembly` two times: with `BACKGROUND = "ecoinvent"` and with `BACKGROUND = "bafu"`
   (or set the environment variable `L4M_BACKGROUND`). The results go to `results/` and `results/bafu/`.
3. Run this notebook.

A ratio above 1 means that BAFU gives a higher impact than ecoinvent.

In [1]:
import bw2data as bd
import pandas as pd
import plotly.express as px
from IPython.display import display

from l4m_lca.compare import compare
from l4m_lca.constants import BACKGROUNDS, JOINING_STEM, MATERIAL_STEM, PREPOST_STEM
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.project import category, ef31_methods, find_activity, short

## Parameters

In [2]:
LABELS = ("ecoinvent", "bafu")
SCENARIO = "amount"
RESULT_SETS = {
    "Joining": JOINING_STEM,
    "Pre-/post-processes": PREPOST_STEM,
    "Material (per kg)": MATERIAL_STEM,
}

# Matching background inputs: label -> {background: (name, location)}
INPUTS = {
    "Electricity MV, RER": {"ecoinvent": ("market group for electricity, medium voltage", "RER"),
                            "bafu": ("Electricity, medium voltage, at grid", "RER")},
    "Electricity MV, NL": {"ecoinvent": ("market for electricity, medium voltage", "NL"),
                           "bafu": ("Electricity, medium voltage, at grid", "NL")},
    "Argon, liquid": {"ecoinvent": ("market for argon, liquid", "RER"), "bafu": ("Argon, liquid, at plant", "RER")},
    "Nitric acid": {"ecoinvent": ("market for nitric acid, without water, in 50% solution state", "RER w/o RU"),
                    "bafu": ("Nitric acid, 50% in H2O, at plant", "RER")},
    "Sodium hydroxide": {"ecoinvent": ("market for sodium hydroxide, without water, in 50% solution state", "RER"),
                         "bafu": ("Sodium hydroxide, 50% in H2O, production mix, at plant", "RER")},
    "Aluminium, wrought": {"ecoinvent": ("market for aluminium, wrought alloy", "GLO"),
                           "bafu": ("Aluminium, production mix, wrought alloy, at plant", "RER")},
    "Steel 18/8": {"ecoinvent": ("market for steel, chromium steel 18/8", "GLO"),
                   "bafu": ("Chromium steel 18/8, at plant", "RER")},
}

## Foreground results: BAFU / ecoinvent

In [3]:
tables = []
for name, stem in RESULT_SETS.items():
    t = compare(BACKGROUNDS[LABELS[0]]["results"], BACKGROUNDS[LABELS[1]]["results"], stem, *LABELS, scenario=SCENARIO)
    tables.append(t.assign(set=name))
comparison = pd.concat(tables, ignore_index=True)
comparison["indicator"] = comparison["indicator"].map(short)
ratios = comparison.pivot_table(index=["set", "Process"], columns="indicator", values="ratio", sort=False)
display(ratios.style.format("{:.2f}").background_gradient(cmap="RdBu_r", vmin=0, vmax=2, axis=None))

In [4]:
impact_ratios = comparison[~comparison["indicator"].str.contains(r"\(kWh\)|\(EUR\)")]
# Colour and symbol per process set (accessible palette of l4m_lca.plots); one row per indicator
fig = px.strip(impact_ratios, x="ratio", y="indicator", color="set", hover_data=["Process"],
               color_discrete_sequence=["#2a78d6", "#eb6834"],
               labels={"ratio": "BAFU result / ecoinvent result", "indicator": "", "set": "Process set"},
               title=f"BAFU / ecoinvent per process ({SCENARIO} scenario); 1 = same result")
fig.update_traces(marker=dict(size=11, line=dict(color="white", width=1)))
for trace, symbol in zip(fig.data, ["circle", "diamond", "square", "triangle-up"]):
    trace.marker.symbol = symbol
fig.add_vline(x=1, line_dash="dash", line_color="#5d6a64", annotation_text="same result",
              annotation_position="top")
fig.update_layout(template="plotly_white", height=520, width=1200, font=dict(size=14, color="#1d2421"))
display(fig)

## Background inputs: impact of 1 unit

This table explains the foreground differences: the foreground amounts are the same in both runs.

In [5]:
unit_rows = []
for label in LABELS:
    profile = BACKGROUNDS[label]
    bd.projects.set_current(profile["project"])
    methods = ef31_methods()
    nodes = {k: find_activity(profile["database"], *v[label]) for k, v in INPUTS.items()}
    impacts = BackgroundImpacts(methods, next(iter(nodes.values())))
    for k, node in nodes.items():
        for m, value in zip(methods, impacts.unit(node)):
            unit_rows.append({"input": k, "indicator": short(category(m)), "background": label, "value": value})
units_df = pd.DataFrame(unit_rows).pivot_table(index=["input", "indicator"], columns="background", values="value", sort=False)
units_df["ratio"] = units_df[LABELS[1]] / units_df[LABELS[0]]
units_df["ratio"].unstack("indicator").style.format("{:.2f}").background_gradient(cmap="RdBu_r", vmin=0, vmax=2, axis=None)

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 3.43e+12)
  warnings.warn(msg, UmfpackWarning)


indicator,acidification,eutrophication: terrestrial,human toxicity: carcinogenic,"human toxicity: carcinogenic, inorganics","human toxicity: carcinogenic, organics",particulate matter formation,photochemical oxidant formation: human health,climate change: total
input,,,,,,,,
"Electricity MV, RER",0.68,0.84,3.84,1.14,8.49,1.40,0.88,0.88
"Electricity MV, NL",0.41,0.46,4.77,1.05,6.79,0.58,0.55,0.61
"Argon, liquid",0.09,0.09,0.67,0.18,1.48,0.15,0.10,0.12
Nitric acid,1.43,1.54,1.38,1.43,1.34,1.85,3.00,2.97
Sodium hydroxide,0.53,0.46,0.15,4.85,0.05,0.42,0.50,0.60
"Aluminium, wrought",0.28,0.25,0.66,2.68,0.15,0.24,0.31,0.38
Steel 18/8,0.79,0.77,1.46,1.76,0.82,0.73,0.87,0.88


## Import quality of BAFU-2026

Exchanges that did not link during the import are removed. Their emissions are missing from the BAFU results.

In [6]:
unlinked = pd.read_csv(f"{BACKGROUNDS['bafu']['results']}/unlinked_exchanges.csv")
display(unlinked.groupby("type")["occurrences"].agg(["count", "sum"]))
unlinked.head(20)

,count,sum
type,,
biosphere,383,10067
technosphere,5,14


,type,name,categories,unit,occurrences
0,biosphere,"Noise, road, passenger car, average",non material emissions,kilometer,620
1,biosphere,"Noise, road, lorry, average",non material emissions,kilometer,503
2,biosphere,"Water, well",natural resource::in water,cubic meter,244
3,biosphere,Sand,natural resource::in ground,kilogram,142
4,biosphere,Potassium chloride,natural resource::in ground,kilogram,141
5,biosphere,Barite,natural resource::in ground,kilogram,140
6,biosphere,Clay,natural resource::in ground,kilogram,138
7,biosphere,"Water, fossil",natural resource::in water,cubic meter,138
8,biosphere,"Ethane, 1,2-dichloro-",water::surface water,kilogram,137
9,biosphere,"Gas, natural/m3",natural resource::in ground,cubic meter,137
